# LoopWise — train `yolo26n-cls` (8 waste classes) on Colab GPU

Reproduces the exact local training path (`ml/train.py`) on a T4 GPU, because the
local laptop (i5-10210U, 7.84 GB RAM, CPU-only torch) measured **0.44 GB min free
RAM** and **~5.5 min/epoch**, i.e. ~4.6 h for 50 epochs with memory thrashing.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Keep the runtime alive
for the whole training (~10–20 min).

Everything below reuses the repo's own scripts and the same hyperparameters as the
local smoke test: `--seed 42 --imgsz 224 --patience 15`, 50 max epochs, deterministic,
val-selected checkpoints, test set used only for the final evaluation.

## 1. GPU check + install (do NOT reinstall torch — Colab's CUDA build must stay)

In [ ]:
import torch, sys
print('python', sys.version.split()[0])
print('torch', torch.__version__, 'cuda_available', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Runtime → Change runtime type → T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))
!pip -q install "ultralytics>=8.4.173" kagglehub
import ultralytics; print('ultralytics', ultralytics.__version__)

## 2. Get the repo
Clone your fork, or upload the repo. Then work from its root.

In [ ]:
import os
REPO_URL = 'https://github.com/OWNER/LoopWise.git'  # <-- set me (or leave empty and upload)
if REPO_URL and not os.path.exists('/content/LoopWise'):
    !git clone --depth 1 $REPO_URL /content/LoopWise
if not os.path.exists('/content/LoopWise'):
    # Private/unavailable repo: upload the repo's `ml/` folder as a zip instead.
    from google.colab import files
    print('Upload a zip of your LoopWise `ml/` folder')
    up = files.upload()
    !unzip -q -o {list(up)[0]} -d /content/LoopWise
os.chdir('/content/LoopWise')
!pwd && ls ml

## 3. Download the Kaggle source dataset (read-only raw input)
`ml/data/raw/` must end up with one folder per SOURCE label:
`battery/ biological/ brown-glass/ cardboard/ clothes/ green-glass/ metal/ paper/ plastic/ shoes/ trash/ white-glass/`

In [ ]:
import kagglehub, os, shutil
src = kagglehub.dataset_download('mostafaabla/garbage-classification')
print('downloaded to', src)
print(sorted(os.listdir(src))[:20])
raw = '/content/LoopWise/ml/data/raw'
os.makedirs(raw, exist_ok=True)
if not os.listdir(raw):
    for item in os.listdir(src):
        s, d = os.path.join(src, item), os.path.join(raw, item)
        (shutil.copytree if os.path.isdir(s) else shutil.copy2)(s, d)
print('raw labels:', sorted(os.listdir(raw)))

## 4. Prepare the 8-class dataset + validate (never modifies raw)

In [ ]:
!python ml/prepare_dataset.py --source-root ml/data/raw --out ml/data/waste-classify \
    --dataset kaggle12 --seed 42 --val-frac 0.1 --test-frac 0.1 --max-per-class 1000
!python ml/validate_dataset.py --data ml/data/waste-classify

## 5. Train
Same script and config as local; `--device 0` for the T4. `--project` is passed as an
**absolute** path so the run lands exactly in `ml/runs/<name>-<UTC>` (a relative path
gets nested under `runs/classify/` by Ultralytics). Writes `best.pt`, `last.pt`,
`results.csv`, `run_record.json`, `settings_used.json`, `train.log`.
50 max epochs, early stopping after 15 epochs without val improvement; checkpoints are
selected on **val** only — `test/` is evaluated once at the end.

> Requires the updated `ml/train.py` from this branch (it adds `--model-version` and
> `run_record.json`). Commit/push before cloning, or upload `ml/` via step 2.

In [ ]:
!python ml/train.py --data ml/data/waste-classify --model yolo26n-cls.pt \
    --epochs 50 --imgsz 224 --batch 64 --device 0 --seed 42 --patience 15 \
    --project /content/LoopWise/ml/runs \
    --name baseline --model-version yolo26n-cls-v1 2>&1 | tee /content/train.log
!ls -R ml/runs | tail -40

## 6. Export the best checkpoint to ONNX
`export_onnx.py` refuses to export if the checkpoint's class order does not match
`ml/data.yaml`, then validates the graph and writes the class-index map.

In [ ]:
import glob, os
runs = sorted(glob.glob('ml/runs/baseline-*'))
best = os.path.join(runs[-1], 'weights', 'best.pt')
print('best checkpoint:', best)
!python ml/export_onnx.py --weights {best} --out ml/models/loopwise-yolo26n-cls.onnx \
    --model-version yolo26n-cls-v1

## 7. Copy the artifacts back into the repo
Then download them to your machine and place them at:
- `ml/runs/<run>/...` for the record (or keep them outside Git — `ml/runs/` is gitignored)
- `ml/models/loopwise-yolo26n-cls.onnx` + `ml/models/loopwise-yolo26n-cls.class-index-map.json`

In [ ]:
import glob, os, shutil, zipfile
runs = sorted(glob.glob('/content/LoopWise/ml/runs/baseline-*'))[-1]
print('run dir:', runs)
for name in ('run_record.json', 'metrics.json', 'settings_used.json', 'results.csv', 'args.yaml'):
    p = os.path.join(runs, name)
    print(name, '->', open(p).read()[:400] if os.path.exists(p) else 'MISSING')
shutil.copy('/content/train.log', os.path.join(runs, 'train.log'))
with zipfile.ZipFile('/content/loopwise_artifacts.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(runs + '/**/*', recursive=True):
        if os.path.isfile(f): z.write(f)
    for f in glob.glob('/content/LoopWise/ml/models/*'):
        if os.path.isfile(f): z.write(f)
from google.colab import files
files.download('/content/loopwise_artifacts.zip')

## 8. Verify what you got
`metrics.json` holds the real, measured numbers (`val_top1/val_top5`, `test_top1/test_top5`)
produced by this run. Do **not** quote accuracy until these files exist — the local smoke
test only proved the pipeline runs, not that the model converges.